# YOLO26n on NEU-DET — Steel Defect Detection

## Goal
Train YOLO26n (latest Ultralytics model, June 2026) on NEU-DET with the same optimized recipe as our YOLO11n baseline.
Target: Beat 78.8% mAP@0.5 (YOLO11n baseline).

## Why YOLO26?
| Feature | YOLO11n | YOLO26n |
|---------|---------|----------|
| COCO mAP | 39.5 | 40.9 (+1.4) |
| NMS-free | ❌ | ✅ |
| Small objects | — | STAL + P2 variant |
| CPU speed | baseline | 43% faster |
| Params | 2.6M | 2.4M |

## Training Recipe (same as YOLO11n baseline)
| Parameter | Value | Rationale |
|-----------|-------|----------|
| mosaic | 0.0 | DISABLED — sabotages fine defect features |
| mixup | 0.15 | Light regularization |
| copy_paste | 0.1 | Copy-paste augmentation |
| optimizer | AdamW | Better for fine-grained features |
| lr0 | 0.001 | Lower LR for AdamW |
| epochs | 600 | More time to converge |
| patience | 150 | More exploration room |
| imgsz | 800 | Good for thin defects |

## Requirements
- ultralytics >= 8.4.0 (YOLO26 support)
- Cell 1 handles the upgrade automatically

In [1]:
# Cell 1: Upgrade Ultralytics & Environment Setup
import subprocess, sys

# Upgrade ultralytics to get YOLO26 support
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'ultralytics>=8.4.0'])
print('Ultralytics upgraded. Restart kernel if prompted.')

import torch
import ultralytics
from pathlib import Path
import json
import os
import random
import numpy as np

# Seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f'Python version:    {sys.version.split()[0]}')
print(f'PyTorch version:   {torch.__version__}')
print(f'Ultralytics ver:   {ultralytics.__version__}')
print(f'CUDA available:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:               {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory:        {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected — training will be extremely slow!')

# Verify YOLO26 is available
try:
    from ultralytics import YOLO
    test = YOLO('yolo26n.yaml')
    print(f'\nYOLO26 support:    OK Available')
    del test
except Exception as e:
    print(f'\nYOLO26 support:    FAIL Not available — {e}')
    print('Try restarting the kernel after upgrade.')

Ultralytics upgraded. Restart kernel if prompted.
Python version:    3.11.15
PyTorch version:   2.6.0+cu124
Ultralytics ver:   8.4.83
CUDA available:    True
GPU:               NVIDIA RTX 2000 Ada Generation
GPU Memory:        17.2 GB

YOLO26 support:    OK Available


In [2]:
# Cell 2: Path Setup & Verification
from pathlib import Path
import yaml

# Paths
ROOT = Path(r'D:/DigiSteel-Yolo/DigiSteel-YOLO')
DATA_YAML = ROOT / 'configs' / 'data' / 'neu_det.yaml'
RUNS_DIR = ROOT / 'runs' / 'detect'
RUN_NAME = 'yolov26n_neudet'
BEST_PT = RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'
RESULTS_CSV = RUNS_DIR / RUN_NAME / 'results.csv'
EVALS_DIR = ROOT / 'evals'
METRICS_JSON = EVALS_DIR / 'yolov26n_neudet_results.json'
TRAIN_STATUS_FILE = EVALS_DIR / 'yolov26n_neudet_train_status.json'

# Verify critical files
print('=' * 60)
print('PATH VERIFICATION')
print('=' * 60)

errors = []
if not DATA_YAML.exists():
    errors.append(f'Data YAML not found: {DATA_YAML}')
else:
    print(f'  Data YAML: {DATA_YAML}')
    with open(DATA_YAML, 'r') as f:
        data_cfg = yaml.safe_load(f)
    base_path = Path(data_cfg.get('path', ''))
    if not base_path.is_absolute():
        base_path = DATA_YAML.parent / base_path
    for split in ['train', 'val', 'test']:
        split_rel = data_cfg.get(split, '')
        if split_rel:
            split_path = base_path / split_rel
            if split_path.exists():
                count = len(list(split_path.glob('*.jpg'))) + len(list(split_path.glob('*.png')))
                print(f'  {split}: {split_path} ({count} images)')
            else:
                errors.append(f'{split} path not found: {split_path}')

EVALS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

if errors:
    print('\nERRORS:')
    for e in errors:
        print(f'  {e}')
    raise FileNotFoundError('Critical paths missing.')

print('\nAll checks passed. Ready to train.')

PATH VERIFICATION
  Data YAML: D:\DigiSteel-Yolo\DigiSteel-YOLO\configs\data\neu_det.yaml
  train: D:\DigiSteel-Yolo\DigiSteel-YOLO\datasets\NEU-DET\yolo\images\train (1290 images)
  val: D:\DigiSteel-Yolo\DigiSteel-YOLO\datasets\NEU-DET\yolo\images\val (344 images)
  test: D:\DigiSteel-Yolo\DigiSteel-YOLO\datasets\NEU-DET\yolo\images\test (166 images)

All checks passed. Ready to train.


In [3]:
# Cell 3: Load YOLO26n Pretrained Model
from ultralytics import YOLO

# Load YOLO26n from pretrained weights (auto-downloads if not local)
model = YOLO('yolo26n.pt')

print('=' * 60)
print('YOLO26n MODEL INFO')
print('=' * 60)
print(f'Architecture:  YOLO26n (pretrained on COCO)')
print(f'Model type:    {type(model.model).__name__}')
print(f'Task:          {model.task}')
print(f'Parameters:    {sum(p.numel() for p in model.model.parameters()):,}')
print(f'Layers:        {len(list(model.model.modules()))}')
print('NMS-free:      Yes (end-to-end by default)')
print('DFL-free:      Yes (lighter regression head)')
print('=' * 60)

YOLO26n MODEL INFO
Architecture:  YOLO26n (pretrained on COCO)
Model type:    DetectionModel
Task:          detect
Parameters:    2,572,280
Layers:        454
NMS-free:      Yes (end-to-end by default)
DFL-free:      Yes (lighter regression head)


In [4]:
# Cell 4: Training — Same Optimized Recipe as YOLO11n Baseline
from ultralytics import YOLO
import time
import json as _json

# Build model (cell-independent)
model = YOLO('yolo26n.pt')

# Training hyperparameters — same recipe as fresh_baseline
base_overrides = {
    'data': str(DATA_YAML),
    'task': 'detect',
    'epochs': 600,
    'patience': 150,
    'batch': 16,
    'imgsz': 800,
    'device': 0,
    'optimizer': 'AdamW',
    'lr0': 0.001,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 5,
    'warmup_momentum': 0.8,
    'warmup_bias_lr': 0.1,
    'mosaic': 0.0,
    'mixup': 0.15,
    'copy_paste': 0.1,
    'copy_paste_mode': 'flip',
    'degrees': 10.0,
    'translate': 0.2,
    'scale': 0.6,
    'shear': 5.0,
    'perspective': 0.0,
    'flipud': 0.5,
    'fliplr': 0.5,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'erasing': 0.4,
    'cos_lr': True,
    'deterministic': True,
    'close_mosaic': 10,
    'amp': True,
    'seed': 42,
    'workers': 8,
    'save': True,
    'save_period': 50,
    'plots': True,
    'project': str(RUNS_DIR),
    'name': RUN_NAME,
    'exist_ok': True,
    'verbose': True,
}

print('=' * 60)
print('TRAINING CONFIGURATION')
print('=' * 60)
print(f'Architecture:  YOLO26n (pretrained)')
print(f'Dataset:       {DATA_YAML}')
print(f'Epochs:        {base_overrides["epochs"]}')
print(f'Patience:      {base_overrides["patience"]}')
print(f'Batch:         {base_overrides["batch"]}')
print(f'Image size:    {base_overrides["imgsz"]}')
print(f'Optimizer:     {base_overrides["optimizer"]}')
print(f'LR:            {base_overrides["lr0"]}')
print(f'Mosaic:        {base_overrides["mosaic"]} (disabled)')
print(f'Mixup:         {base_overrides["mixup"]}')
print(f'Copy-paste:    {base_overrides["copy_paste"]} ({base_overrides["copy_paste_mode"]})')
print(f'Output:        {RUNS_DIR / RUN_NAME}')
print('=' * 60)
print()

# Train with copy_paste, fallback to 0.0 if it fails
training_start = time.time()
copy_paste_used = base_overrides['copy_paste']

try:
    print(f'Starting training with copy_paste={copy_paste_used}...')
    results = model.train(**base_overrides)
except Exception as e:
    error_msg = str(e).lower()
    if 'copy_paste' in error_msg or 'segment' in error_msg or 'mask' in error_msg:
        print(f'\ncopy_paste={copy_paste_used} failed. Retrying without it...')
        print(f'Error: {e}')
        base_overrides.pop('copy_paste', None)
        copy_paste_used = 0.0
        model = YOLO('yolo26n.pt')
        results = model.train(**base_overrides)
    else:
        raise

training_time = time.time() - training_start

# Save training status
_train_status = {
    'training_time_seconds': training_time,
    'copy_paste_used': copy_paste_used,
    'model': 'yolo26n.pt',
}
with open(TRAIN_STATUS_FILE, 'w') as _f:
    _json.dump(_train_status, _f, indent=2)

print(f'\n{"=" * 60}')
print('TRAINING COMPLETE')
print(f'{"=" * 60}')
print(f'Duration: {training_time / 3600:.1f} hours')
print(f'Copy-paste: {copy_paste_used}')
print(f'Weights: {BEST_PT}')
print('=' * 60)

TRAINING CONFIGURATION
Architecture:  YOLO26n (pretrained)
Dataset:       D:\DigiSteel-Yolo\DigiSteel-YOLO\configs\data\neu_det.yaml
Epochs:        600
Patience:      150
Batch:         16
Image size:    800
Optimizer:     AdamW
LR:            0.001
Mosaic:        0.0 (disabled)
Mixup:         0.15
Copy-paste:    0.1 (flip)
Output:        D:\DigiSteel-Yolo\DigiSteel-YOLO\runs\detect\yolov26n_neudet

Starting training with copy_paste=0.1...
Ultralytics 8.4.83  Python-3.11.15 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\DigiSteel-Yolo\DigiSteel-YOLO\configs\data\neu_det.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropo

In [5]:
# Cell 5: Evaluate on Test Set
from ultralytics import YOLO
import json
from pathlib import Path

# Load training status if available
import json as _json
if TRAIN_STATUS_FILE.exists():
    with open(TRAIN_STATUS_FILE) as _f:
        _status = _json.load(_f)
    training_time = _status.get('training_time_seconds', 0)
    copy_paste_used = _status.get('copy_paste_used', 0.1)
else:
    training_time = 0
    copy_paste_used = 0.1

# Find best weights
if not BEST_PT.exists():
    candidates = list((RUNS_DIR / RUN_NAME).rglob('best.pt'))
    if candidates:
        BEST_PT = candidates[0]
    else:
        raise FileNotFoundError(f'No best.pt found in {RUNS_DIR / RUN_NAME}')

print(f'Loading best weights: {BEST_PT}')
best_model = YOLO(str(BEST_PT))

# Evaluate on test set
print(f'\nRunning validation on test set...')
val_results = best_model.val(
    data=str(DATA_YAML),
    split='test',
    imgsz=800,
    batch=16,
    device=0,
    plots=True,
    verbose=True,
)

# Extract metrics
map50 = float(val_results.box.map50)
map50_95 = float(val_results.box.map)
precision = float(val_results.box.mp)
recall = float(val_results.box.mr)

# Per-class AP@0.5
class_names = getattr(val_results, 'names', None) or best_model.names
per_class_ap50 = {}
for cls_id, cls_name in class_names.items():
    if cls_id < len(val_results.box.ap50):
        per_class_ap50[cls_name] = float(val_results.box.ap50[cls_id])

print(f'\n{"=" * 60}')
print('TEST SET RESULTS — YOLO26n')
print('=' * 60)
print(f'mAP@0.5:      {map50:.4f} ({map50*100:.1f}%)')
print(f'mAP@0.5:0.95: {map50_95:.4f} ({map50_95*100:.1f}%)')
print(f'Precision:    {precision:.4f}')
print(f'Recall:       {recall:.4f}')
print(f'\nPer-class AP@0.5:')
print('-' * 40)
for cls_name, ap in per_class_ap50.items():
    print(f'  {cls_name:<20} {ap:.4f} ({ap*100:.1f}%)')
print('=' * 60)

# Compare with YOLO11n baseline
baseline_json = EVALS_DIR / 'fresh_baseline_results.json'
if baseline_json.exists():
    with open(baseline_json) as f:
        baseline = json.load(f)
    print(f'\n{"=" * 60}')
    print('COMPARISON vs YOLO11n Baseline')
    print('=' * 60)
    b_map50 = baseline['map50']
    sign = '+' if map50 > b_map50 else ''
    print(f'mAP@0.5:  {b_map50*100:.1f}% -> {map50*100:.1f}%  ({sign}{(map50 - b_map50)*100:.1f}%)')
    b_map5095 = baseline['map50_95']
    sign2 = '+' if map50_95 > b_map5095 else ''
    print(f'mAP@50:95:{b_map5095*100:.1f}% -> {map50_95*100:.1f}%  ({sign2}{(map50_95 - b_map5095)*100:.1f}%)')
    print(f'\nPer-class delta:')
    for cls_name, ap in per_class_ap50.items():
        b_ap = baseline['per_class_ap50'].get(cls_name, 0)
        delta = ap - b_ap
        ds = '+' if delta > 0 else ''
        print(f'  {cls_name:<20} {b_ap*100:.1f}% -> {ap*100:.1f}%  ({ds}{delta*100:.1f}%)')
    print('=' * 60)

# Save metrics
metrics = {
    'experiment': 'yolov26n_neudet',
    'model': 'yolo26n.pt',
    'map50': map50,
    'map50_95': map50_95,
    'precision': precision,
    'recall': recall,
    'per_class_ap50': per_class_ap50,
    'training_time_hours': round(training_time / 3600, 2),
    'copy_paste_used': copy_paste_used,
    'hyperparameters': {
        'epochs': 600, 'patience': 150, 'batch': 16, 'imgsz': 800,
        'optimizer': 'AdamW', 'lr0': 0.001, 'mosaic': 0.0,
        'mixup': 0.15, 'copy_paste': copy_paste_used, 'cos_lr': True,
    },
}

with open(METRICS_JSON, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'\nMetrics saved to: {METRICS_JSON}')

Loading best weights: D:\DigiSteel-Yolo\DigiSteel-YOLO\runs\detect\yolov26n_neudet\weights\best.pt

Running validation on test set...
Ultralytics 8.4.83  Python-3.11.15 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
YOLO26n summary (fused): 122 layers, 2,376,006 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.30.3 ms, read: 0.50.2 MB/s, size: 18.9 KB)
val: Scanning D:\DigiSteel-Yolo\DigiSteel-YOLO\datasets\NEU-DET\yolo\labels\test.cache... 166 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 166/166  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 2.3it/s 4.8s0.2s
                   all        166        371      0.773      0.762       0.81      0.431
               crazing         24         54      0.576      0.519      0.551       0.16
             inclusion         33         77      0.783      0.818      0.847      0.416
               patches         31         80  

In [6]:
# Cell 6: Training Curves Visualization
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

if not RESULTS_CSV.exists():
    alt = RUNS_DIR / RUN_NAME / 'results.csv'
    if alt.exists():
        RESULTS_CSV = alt
    else:
        raise FileNotFoundError('results.csv not found')

df = pd.read_csv(RESULTS_CSV)
df.columns = df.columns.str.strip()
print(f'Loaded results: {len(df)} epochs')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('YOLO26n on NEU-DET — Training Curves', fontsize=14, fontweight='bold')

# Loss
loss_cols = [c for c in df.columns if 'loss' in c.lower()]
if loss_cols:
    ax = axes[0, 0]
    for col in loss_cols:
        ax.plot(df[col], label=col, alpha=0.8)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training and Validation Loss')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# mAP
map50_col = [c for c in df.columns if 'map50' in c.lower() and 'map50-' not in c.lower()]
map_col = [c for c in df.columns if 'map50-95' in c.lower()]
ax = axes[0, 1]
if map50_col:
    ax.plot(df[map50_col[0]], label='mAP@0.5', color='blue', linewidth=2)
if map_col:
    ax.plot(df[map_col[0]], label='mAP@0.5:0.95', color='red', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('mAP')
ax.set_title('mAP Progress')
ax.legend()
ax.grid(True, alpha=0.3)

# Learning Rate
lr_col = [c for c in df.columns if 'lr' in c.lower() or 'pg0' in c.lower()]
ax = axes[1, 0]
if lr_col:
    ax.plot(df[lr_col[0]], color='green', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning Rate')
    ax.set_title('Learning Rate Schedule')
    ax.grid(True, alpha=0.3)

# Precision & Recall
prec_col = [c for c in df.columns if 'precision' in c.lower()]
rec_col = [c for c in df.columns if 'recall' in c.lower()]
ax = axes[1, 1]
if prec_col:
    ax.plot(df[prec_col[0]], label='Precision', color='purple', linewidth=2)
if rec_col:
    ax.plot(df[rec_col[0]], label='Recall', color='orange', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Score')
ax.set_title('Precision and Recall')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RUNS_DIR / RUN_NAME / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Curves saved to: {RUNS_DIR / RUN_NAME / "training_curves.png"}')

Loaded results: 452 epochs


<Figure size 1400x1000 with 4 Axes>

Curves saved to: D:\DigiSteel-Yolo\DigiSteel-YOLO\runs\detect\yolov26n_neudet\training_curves.png


In [7]:
# Cell 7: Per-Class AP Comparison Bar Chart
import matplotlib.pyplot as plt
import numpy as np
import json

# Load results
with open(METRICS_JSON) as f:
    yolo26_metrics = json.load(f)

baseline_json = EVALS_DIR / 'fresh_baseline_results.json'
if baseline_json.exists():
    with open(baseline_json) as f:
        baseline_metrics = json.load(f)

    classes = list(yolo26_metrics['per_class_ap50'].keys())
    yolo26_aps = [yolo26_metrics['per_class_ap50'][c] * 100 for c in classes]
    baseline_aps = [baseline_metrics['per_class_ap50'][c] * 100 for c in classes]

    x = np.arange(len(classes))
    width = 0.35

    fig, ax = plt.subplots(figsize=(12, 6))
    bars1 = ax.bar(x - width/2, baseline_aps, width, label='YOLO11n (baseline)', color='#4ECDC4', edgecolor='black', linewidth=0.5)
    bars2 = ax.bar(x + width/2, yolo26_aps, width, label='YOLO26n', color='#FF6B6B', edgecolor='black', linewidth=0.5)

    ax.set_ylabel('AP@0.5 (%)', fontsize=12)
    ax.set_title('Per-Class AP@0.5: YOLO11n vs YOLO26n on NEU-DET', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(classes, rotation=30, ha='right', fontsize=10)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 105)

    for bar in bars1:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 1, f'{h:.1f}', ha='center', va='bottom', fontsize=8)
    for bar in bars2:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 1, f'{h:.1f}', ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    plt.savefig(RUNS_DIR / RUN_NAME / 'per_class_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Baseline results not found — skipping comparison chart.')
    print('Run the fresh_baseline notebook first for comparison.')

<Figure size 1200x600 with 1 Axes>